# neo4j from Python — StatBytes demo notebook

Two sections: the Cypher queries from `cypher/02_basics.cypher`, then the Graph Data Science steps from `cypher/03_gds.cypher`. Every call goes through `gds.run_cypher`, which returns a pandas DataFrame.

Needs the container from `docker compose up -d` and the data loaded by `cypher/01_load.cypher`. Packages: `pip install graphdatascience neo4j pandas`.

In [1]:
from graphdatascience import GraphDataScience
import pandas as pd

gds = GraphDataScience("bolt://localhost:7687", auth=("neo4j", "statbytes2026"))
q = gds.run_cypher      # q("MATCH ...") -> DataFrame

q("RETURN gds.version() AS gds_version")

,gds_version
0,2.13.14


## Part 2 — Cypher

**Q1.** What's in the database?

In [ ]:
q("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC")

In [ ]:
q("MATCH ()-[r]->() RETURN type(r) AS relationship, count(*) AS n ORDER BY n DESC")

**Q2.** Look up one node. (Compare: `SELECT * FROM substation WHERE name = 'Allegheny-01'`.) Returning the node gives a dict of its properties; returning the properties directly gives columns.

In [ ]:
q("MATCH (s:Substation {name: 'Allegheny-01'}) RETURN s.name, s.voltage_kv, s.region, s.online")

**Q3.** One hop out: the neighbours of that substation and the line properties.

In [ ]:
q("""
MATCH (s:Substation {name: 'Allegheny-01'})-[r:LINE]-(t:Substation)
RETURN t.name AS neighbour, t.voltage_kv AS kv, r.length_km AS km, r.fault_prob AS fault_prob
ORDER BY km
""")

**Q4.** A longer pattern: nuclear plants, their substations, and what's one line away.

In [ ]:
q("""
MATCH (g:Generator {type: 'nuclear'})-[:FEEDS]->(s:Substation)-[:LINE]-(t:Substation)
RETURN g.name AS plant, s.name AS substation, collect(t.name) AS one_line_away
""")

**Q5.** Filter and aggregate: online generation vs demand, by region.

In [ ]:
q("""
MATCH (g:Generator)-[:FEEDS]->(s:Substation)
WHERE g.online
WITH s.region AS region, sum(g.capacity_mw) AS generation_mw
MATCH (t:Substation {region: region})-[:SUPPLIES]->(l:Load)
RETURN region, generation_mw, sum(l.demand_mw) AS demand_mw
ORDER BY region
""")

**Q6.** Variable-length paths: hospitals within 3 lines of a nuclear plant.

In [ ]:
q("""
MATCH (g:Generator {type: 'nuclear'})-[:FEEDS]->(:Substation)-[:LINE*1..3]-(s:Substation)
      -[:SUPPLIES]->(h:Load {kind: 'hospital'})
RETURN DISTINCT g.name AS plant, h.name AS hospital, s.name AS via_substation
""")

**Q7.** Shortest path between two distant substations.

In [ ]:
q("""
MATCH (a:Substation {name: 'Allegheny-01'}), (b:Substation {name: 'Pocono-01'})
MATCH p = shortestPath((a)-[:LINE*]-(b))
RETURN length(p) AS hops, [n IN nodes(p) | n.name] AS route
""")

**Q8.** A write. Parameters (`$name`) are the normal way to pass values from Python.

In [ ]:
q("MATCH (s:Substation {name: $name}) SET s.inspected = date() RETURN s.name, s.inspected",
  {"name": "Allegheny-01"})

## Part 3 — Graph Data Science

**G0.** Project the substation/line graph into GDS memory.

In [ ]:
for name in ["grid", "grid_online"]:                 # drop leftovers from an earlier run
    q(f"CALL gds.graph.drop('{name}', false) YIELD graphName")

q("""
CALL gds.graph.project('grid', 'Substation',
  { LINE: { orientation: 'UNDIRECTED', properties: ['length_km', 'capacity_mw'] } })
YIELD graphName, nodeCount, relationshipCount
""")

**G1.** Degree centrality.

In [ ]:
q("""
CALL gds.degree.stream('grid') YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS s, score
RETURN s.name AS substation, s.region AS region, s.voltage_kv AS kv, toInteger(score) AS degree
ORDER BY score DESC, s.name LIMIT 10
""")

**G2.** Eigenvector centrality.

In [ ]:
q("""
CALL gds.eigenvector.stream('grid', { maxIterations: 200 }) YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS s, score
RETURN s.name AS substation, s.region AS region, s.voltage_kv AS kv, round(score, 3) AS eigenvector
ORDER BY score DESC LIMIT 10
""")

**G3.** Betweenness centrality.

In [ ]:
q("""
CALL gds.betweenness.stream('grid') YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS s, score
RETURN s.name AS substation, s.region AS region, s.voltage_kv AS kv, round(score) AS betweenness
ORDER BY score DESC LIMIT 10
""")

**G4.** What if a substation broke? Take Susquehanna-27 offline, project only online substations, and find the connected components.

In [ ]:
q("MATCH (s:Substation {name: 'Susquehanna-27'}) SET s.online = false")
q("""
MATCH (s:Substation)-[:LINE]-(t:Substation)
WHERE s.online AND t.online
WITH gds.graph.project('grid_online', s, t) AS g
RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS relationships
""")

In [ ]:
q("""
CALL gds.wcc.stream('grid_online') YIELD nodeId, componentId
WITH gds.util.asNode(nodeId) AS s, componentId
CALL (s) { MATCH (s)<-[:FEEDS]-(g:Generator) WHERE g.online RETURN sum(g.capacity_mw) AS gen }
CALL (s) { MATCH (s)-[:SUPPLIES]->(l:Load)
       RETURN sum(l.demand_mw) AS dem, sum(CASE WHEN l.kind = 'hospital' THEN 1 ELSE 0 END) AS hosp }
RETURN componentId, count(s) AS substations, collect(DISTINCT s.region) AS regions,
       sum(gen) AS generation_mw, sum(dem) AS demand_mw, sum(hosp) AS hospitals
ORDER BY substations DESC
""")

**G5.** Write a result back to the database so ordinary Cypher can use it.

In [ ]:
q("CALL gds.betweenness.write('grid', { writeProperty: 'betweenness' }) YIELD nodePropertiesWritten")
q("MATCH (s:Substation) RETURN s.name, s.region, round(s.betweenness) AS betweenness ORDER BY s.betweenness DESC LIMIT 5")

## Cleanup

**Run last.** `gds.close()` shuts the connection; after this, re-run the first cell before anything else.

In [ ]:
for name in ["grid", "grid_online"]:
    q(f"CALL gds.graph.drop('{name}', false) YIELD graphName")
q("MATCH (s:Substation) SET s.online = true")
gds.close()